# py-URD — function-by-function R⇄Py parity

## 1. Setup

In [1]:
import os, json, sys
from pathlib import Path
import numpy as np
NB = Path('.').resolve(); PORT = NB.parent if NB.name=='examples' else NB
sys.path.insert(0, str(PORT)); sys.path.insert(0, str(PORT.parent/'omicverse-rebuildr'/'engine'))
sys.path.insert(0, str(PORT.parent/'py-destiny'))
import pyurd
from parity_metrics import compute_parity
ref = json.loads((PORT/'data'/'reference_output.json').read_text())
cand = json.loads((PORT/'data'/'candidate_output.json').read_text())

## 2.1 `floodBuildTM`

| R | Python | Default | Description |
|---|---|---|---|
| `object` | `urd` | — | URD object |
| `dm` | `dm_transitions` | — | raw transition matrix |

```r
tm <- URD::floodBuildTM(urd)
```
→ scales transitions so max row-sum = 1.

## 2.2 `floodPseudotime`

| R | Python | Default | Description |
|---|---|---|---|
| `object` | `urd_or_tm` | — | URD or transition matrix |
| `root.cells` | `root_cells` | — | starting cell names/indices |
| `n` | `n` | 20 | number of simulations |
| `minimum.cells.flooded` | `minimum_cells_flooded` | 2 | per-step stop threshold |
| `tm.flood` | `tm_flood` | None | precomputed normalised tm |
| (set.seed) | `seed` | None | master RNG seed |

```r
floods <- URD::floodPseudotime(urd, root.cells=root_ids, n=50, minimum.cells.flooded=1)
```

## 2.3 `floodPseudotimeProcess`

| R | Python | Default | Description |
|---|---|---|---|
| `floods` | `floods` | — | data.frame n_cells × n_sims |
| `max.frac.NA` | `max_frac_NA` | 0.4 | drop cells with > this NA fraction |
| `stability.div` | `stability_div` | 10 | number of cumulative cuts for stability |

```r
urd <- URD::floodPseudotimeProcess(urd, floods, max.frac.NA=0.9, stability.div=5)
```

In [2]:
def to_arr(xs): return np.array([np.nan if x is None else float(x) for x in xs])
pt_r = to_arr(ref['pseudotime']); pt_p = to_arr(cand['pseudotime'])
from scipy.stats import spearmanr, pearsonr
mask = ~np.isnan(pt_r) & ~np.isnan(pt_p)
rho = spearmanr(pt_r[mask], pt_p[mask])[0]
pe = pearsonr(pt_r[mask], pt_p[mask])[0]
print(f'pseudotime Spearman: {rho:.4f}  Pearson: {pe:.4f}')
print(f'  → {"PASS" if rho > 0.80 else "FAIL"}')

pseudotime Spearman: 0.9848  Pearson: 0.9921
  → PASS


## 2.4 `simulateRandomWalksFromTips`

| R | Python | Default | Description |
|---|---|---|---|
| `object` | `urd` | — | URD object |
| `tip.group.size` | `tip_cells` | — | dict tip_name → cell IDs |
| `n.per.tip` | `n_per_tip` | 10000 | walks per tip |
| `root.visits` | (implicit via root_threshold) | — | walk-stop signal |

```r
walks <- URD::simulateRandomWalksFromTips(urd, tip.group.size=5)
```

v0.1 simplification: we drive walks only by pseudotime-direction (no distance-weighted re-sampling).

## 2.5 `processRandomWalks`

Collapses per-walk visitation tables to mean visit frequency per cell per tip.

## 3. Aggregate verdict

| Function | Output | Metric | Pass |
|---|---|---|---|
| `floodPseudotime` + `floodPseudotimeProcess` | pseudotime | Spearman 0.985 | ✅ |
| `simulateRandomWalksFromTips` + `processRandomWalks` | visit-freq | (smoke test — distributional, not exact) | (qualitative) |